# Snapchat Spotlight — Synthetic Experiment Data Generation

This notebook creates the authoritative synthetic source data for the portfolio project. It intentionally implements the treatment effects rather than merely declaring unused parameters.

## Business Scenario

We model a recommendation experiment comparing `Control_Standard_Recs` with `Variant_Algorithmic_Exploration`. The synthetic treatment is designed to increase short-term sharing while introducing a configurable D28 retention penalty and a negative-feedback guardrail effect. These are portfolio modeling assumptions, not claims about Snap production behavior.

In [ ]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

RANDOM_SEED = 42
PROJECT_ID = "driiiportfolio"
DATASET_ID = "snap_spotlight_growth"
NUM_USERS = 25_000
NUM_SESSIONS = 150_000
START_DATE = datetime(2026, 8, 1)

np.random.seed(RANDOM_SEED)


## Treatment Parameters

The parameters below are explicit synthetic assumptions. `d28_treatment_effect` is implemented in the Day-28 return probability.

In [ ]:
SHARES_TREATMENT_EFFECT = 0.35
D28_TREATMENT_EFFECT = -0.025
DISSATISFACTION_BASE_RATE = 0.03
VARIANT_DISSATISFACTION_MULT = 1.85
CONTROL_D28_RATE = 0.80
CONTROL_D7_RATE = 0.70
VARIANT_D7_DELTA = 0.00

assert -1 < D28_TREATMENT_EFFECT < 1


## Generate Users

In [ ]:
user_ids = [f"usr_{i:06d}" for i in range(1, NUM_USERS + 1)]
cohort_dates = [
    (START_DATE + timedelta(days=int(np.random.exponential(scale=7)))).date()
    for _ in range(NUM_USERS)
]
device_os = np.random.choice(["iOS", "Android"], NUM_USERS, p=[0.58, 0.42])
region = np.random.choice(["US_WEST", "US_EAST", "EU_WEST", "APAC"], NUM_USERS, p=[0.40, 0.30, 0.18, 0.12])
experiment_group = np.random.choice(
    ["Control_Standard_Recs", "Variant_Algorithmic_Exploration"],
    NUM_USERS, p=[0.50, 0.50]
)

dim_users = pd.DataFrame({
    "user_id": user_ids,
    "cohort_date": cohort_dates,
    "device_os": device_os,
    "region": region,
    "experiment_group": experiment_group
})

dim_users.head()


## Generate Session Events

The generator first creates a guaranteed onboarding session for every user, then samples additional sessions until the requested target is reached. Day-specific return events are generated independently so exact-day retention is measurable.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
user_index = dim_users.set_index("user_id")

records = []

def add_session(uid, day_offset, group, is_retention_event=False):
    cohort = user_index.loc[uid, "cohort_date"]
    ts = pd.Timestamp(cohort) + pd.Timedelta(days=int(day_offset)) + pd.Timedelta(hours=float(rng.uniform(0, 24)))
    variant = group == "Variant_Algorithmic_Exploration"
    completion = rng.beta(2, 4.8 if variant else 5)
    latency = max(50, int(rng.normal(218 if variant else 210, 35)))
    lam = 0.85 * (1 + SHARES_TREATMENT_EFFECT) if variant else 0.85
    shares = int(rng.poisson(lam))
    neg_prob = DISSATISFACTION_BASE_RATE * (VARIANT_DISSATISFACTION_MULT if variant else 1.0)
    negative = int(rng.random() < neg_prob)
    records.append({
        "session_id": f"sess_{len(records)+1:08d}",
        "user_id": uid,
        "session_timestamp": ts,
        "avg_watch_completion": round(float(np.clip(completion, 0.05, 1.0)), 4),
        "avg_latency_ms": latency,
        "total_shares": shares,
        "negative_feedback_flag": negative
    })

# One onboarding session per user.
for uid, row in dim_users.set_index("user_id").iterrows():
    add_session(uid, 0, row["experiment_group"])

# Exact-day retention probabilities.
for _, row in dim_users.iterrows():
    uid = row.user_id
    group = row.experiment_group
    d7_p = CONTROL_D7_RATE + VARIANT_D7_DELTA if group == "Variant_Algorithmic_Exploration" else CONTROL_D7_RATE
    d28_p = CONTROL_D28_RATE + (D28_TREATMENT_EFFECT if group == "Variant_Algorithmic_Exploration" else 0)
    if rng.random() < d7_p:
        add_session(uid, 7, group, True)
    if rng.random() < d28_p:
        add_session(uid, 28, group, True)

# Fill remaining volume with ordinary sessions on Days 1–27.
while len(records) < NUM_SESSIONS:
    row = dim_users.iloc[int(rng.integers(0, len(dim_users)))]
    add_session(row.user_id, int(rng.integers(1, 28)), row.experiment_group)

fact_user_sessions = pd.DataFrame(records[:NUM_SESSIONS])
fact_user_sessions["session_timestamp"] = pd.to_datetime(fact_user_sessions["session_timestamp"])

print(dim_users.shape)
print(fact_user_sessions.shape)


## Local Validation

In [ ]:
assert len(dim_users) == NUM_USERS
assert len(fact_user_sessions) == NUM_SESSIONS
assert dim_users.user_id.is_unique
assert fact_user_sessions.session_id.is_unique
assert fact_user_sessions.user_id.isin(dim_users.user_id).all()
assert fact_user_sessions.avg_watch_completion.between(0, 1).all()
assert (fact_user_sessions.avg_latency_ms >= 0).all()
assert (fact_user_sessions.total_shares >= 0).all()
assert fact_user_sessions.negative_feedback_flag.isin([0,1]).all()
print("Validation passed.")


## Export CSV Artifacts

In [ ]:
os.makedirs("data", exist_ok=True)
dim_users.to_csv("data/dim_users.csv", index=False)
fact_user_sessions.to_csv("data/fact_user_sessions.csv", index=False)
print("Wrote data/dim_users.csv and data/fact_user_sessions.csv")


## BigQuery Upload — Explicit Schema

This cell is optional. It avoids the prior `REQUIRED → NULLABLE` failure by supplying the exact BigQuery schema and using `WRITE_TRUNCATE`.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

dim_schema = [
    bigquery.SchemaField("user_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("cohort_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("device_os", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("region", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("experiment_group", "STRING", mode="REQUIRED"),
]

session_schema = [
    bigquery.SchemaField("session_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("user_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("session_timestamp", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("avg_watch_completion", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("avg_latency_ms", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("total_shares", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("negative_feedback_flag", "INT64", mode="REQUIRED"),
]

dim_users["cohort_date"] = pd.to_datetime(dim_users["cohort_date"]).dt.date

client.load_table_from_dataframe(
    dim_users,
    f"{PROJECT_ID}.{DATASET_ID}.dim_users",
    job_config=bigquery.LoadJobConfig(schema=dim_schema, write_disposition="WRITE_TRUNCATE")
).result()

client.load_table_from_dataframe(
    fact_user_sessions,
    f"{PROJECT_ID}.{DATASET_ID}.fact_user_sessions",
    job_config=bigquery.LoadJobConfig(schema=session_schema, write_disposition="WRITE_TRUNCATE")
).result()

print("BigQuery load completed with explicit schemas.")
